# Linear Regression (線性迴歸)

## Housing 資料集
此次使用的檔案為房價資料<br>
housing_train.csv 與 housing_test.csv 兩個檔案中紀錄的資料說明如下:<br>
* 第一欄 price 為房屋價格
* 第二欄 area 為房屋的大小
* 第三欄 bedrooms 為臥室的數量
* 第四欄 bathrooms 為浴室的數量
* 第五欄 parking 為停車位的數量

我們希望將第2,3,4,5欄的房屋特徵當作X；第1欄的房價當作Y，找出線性方程式
$$y=\theta^Tx$$
來預測房價。
接下來我們會一步一步完成一個線性迴歸模型，輸入大小、臥室數、浴室數、停車位數給該模型，模型便會預測該房屋的售價。有幾點需要同學注意:<br>
1. **以下程式區塊中，有🚧符號的地方需要同學們自己撰寫或修改程式碼，同學可以新增多行程式碼來完成功能**
2. **程式碼中原始的函數名稱(def 後面的名稱)請不要更改，可能會造成後續的程式無法運行**
3. **過程中會要求同學完成一些python函數，當中的參數多為矩陣形式，shape為[n,m]，當中的n代表的是資料的數量**


## import 必要套件

In [52]:
import numpy as np
import pandas as pd

## 載入訓練資料與測試資料 (3分)
利用 pandas 載入資料<br>
分別將訓練資料與測試資料儲存在 train_data 與 test_data 兩個變數中。<br>
同時我們print出兩筆資料的shape，了解資料在各個維度的尺寸。

**Hint: 用 pandas 載入兩個檔案的資料並將其轉換成 numpy 的形式儲存到 train_data 以及 test_data 中，最終train_data 的shape應該為[380,4]，test_data 為[80,3]**

In [53]:
train_data = pd.read_csv('housing_train.csv')
test_data =  pd.read_csv('housing_test.csv')

train_data = train_data.to_numpy()
test_data  = test_data.to_numpy()

print(train_data.shape)
print(test_data.shape)

(380, 5)
(80, 5)


## Normalization (正規化) (3分)
在開始前我們必須將資料 normalize (正規化)後得到$\hat{x}$。標準化的公式如下:
$$
\hat{x}=\frac{x-mean(x)}{std(x)}
$$
簡單來說，就是將原始資料先減去該資料的平均值，再除以該資料的標準差。<br>
資料正規化有以下好處:
* 正規化的資料大多數會座落在-1~1之間，之後計算 MSE 數值可以收斂在相對小的範圍內，更容易觀察出模型差異
* 每個特徵的數值大都座落在-1~1之間，不會有特徵的原始數值級距差異太大導致模型參數過大

**具體作法**<br>
以本次使用的資料為例，我們將大小、臥室數、浴室數、停車位數這些特徵分別以 $x_1$, $x_2$, $x_3$, $x_4$ 表示，而輸出資料的售價以$y$來表示；
每一個特徵跟輸出都要分別計算正規化，如下:
$$
i\in\{ 1, 2, 3, 4\}, \quad 
\hat{x_{i}}=\frac{x_i-mean(x_i)}{std(x_i)}, \quad
\hat{y}=\frac{y-mean(y)}{std(y)}
$$

由於我們的資料又分為訓練跟測試兩筆不同的資料，在實作上我們只計算訓練資料的平均值跟標準差，等到進行測試時，再使用該平均值跟標準差來標準化測試資料。

**Hint: 使用np.mean()與np.std()函數可以計算出平均值與標準差**

In [54]:
# 🚧 資料正規化
train_mean = np.mean(train_data, axis=0)
train_std  = np.std(train_data, axis=0)

print(train_mean.shape)
print(train_std.shape)

train_data = (train_data-train_mean)/train_std
test_data = (test_data-train_mean)/train_std

(5,)
(5,)


## 分離輸入特徵(x)與輸出資料(y)
將訓練資料與測試資料中的x(第2,3,4,5欄)與y(第1欄)資料分離
分離候用以下變數儲存：
* 訓練資料(共有N筆)
    * x_train: 訓練資料的輸入特徵，尺寸為[N,4]
    * y_train: 訓練資料的輸出，將最後的尺寸調整為[N, 1]，以利後面的運算
* 測試資料(共有U筆)
    * x_test: 測試資料的輸入特徵，尺寸為[U,4]
    * y_test: 測試資料的輸出，將最後的尺寸調整為[U, 1]，以利後面的運算

In [55]:
# 🚧 分離訓練資料
x_train = train_data[:, 1:]
y_train = train_data[:, 0]
y_train = y_train.reshape(-1,1)


# 🚧 分離測試資料
x_test = test_data[:, 1:]
y_test = test_data[:, 0]
y_test = y_test.reshape(-1,1)

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(380, 4)
(380, 1)
(80, 4)
(80, 1)


## 對 x 資料新增常數項 (4分)
除了資料中包含的原始兩項特徵之外，我們需要額外新增一個數值為1的特徵，使我們的線性方程式包含常數項。若我們總共有N筆資料，具體做法為:<br>
1. 原始 *x* 資料尺寸為 [Nx4]
2. 新增一個尺寸為 [Nx1] 且數值皆為1的向量
3. 使用 ***np.concatenate*** 函數將 *x* 與其拼合成一筆 [Nx5] 的資料

In [56]:
# 🚧 對訓練資料的x新增數值為1的向量
ones = np.ones([x_train.shape[0], 1])
x_train = np.concatenate([x_train, ones], axis=1)


# 🚧 對測試資料的x新增數值為1的向量
ones = np.ones([x_test.shape[0],1])
x_test = np.concatenate([x_test, ones],axis=1)

print(x_train.shape)
print(x_test.shape)

(380, 5)
(80, 5)


## 定義模型 (3分)
線性函數:
$$x^i\in R^{5\times 1}, \theta \in R^{5\times 1}, z^i \in R^{1}$$
$$z^i= \theta^Tx^i$$
線性函數(矩陣形式):
$$X\in R^{N\times 5}, \theta \in R^{5\times 1}, Z \in R^{N \times 1}$$
$$Z = X\theta$$

**Hint: 在下方我們會定義 python 的函數 linear_model(x, theta)。**<br>
**參數x的shape為[n, 5]，n代表多筆資料的數量，5代表4個特徵+1個常數項。**<br>
**theta是線性模型的參數，shape為[5,1]，這個尺寸也是因為4個特徵跟1個常數項所致。**

In [57]:
def linear_model(x, theta):
    z=x@theta # 🚧 在這行定義線性函數，使用矩陣形式的乘法
    return z

## 定義 Loss Function (損失函數) (3分)
在 Linear Regression 中:
$$h(x^i)=\theta^Tx^i$$
損失函數 (MSE): 
$$L(\theta)=\frac{1}{N}\sum_{i=1}^{N}(y^i-\theta^Tx^i)^2$$

In [58]:
def get_loss(y_, y):
    loss =  np.mean((y-y_)**2) 
    return loss
    

## Normal Equation (4分)
這邊我們使用線性迴歸的解析解，Normal Equation:
$$
\theta = (X^TX)^{-1}X^TY
$$
切記此公式中標記的各項是使用矩陣形式，在這份作業中:
$$X\in R^{N\times 5}, \theta \in R^{5\times 1}, Z \in R^{N \times 1}$$
**Hint: 矩陣的反矩陣可以使用np.linalg.inv()求出，但有時會遇到無法求出反矩陣的情況，這時要使用np.linalg.pinv()**

In [59]:
def fit(x, y):
    theta = np.linalg.pinv(x.T@x)@x.T@y
    return theta

##### 使用 fit()函數計算出模型參數 theta

In [60]:
theta = fit(x_train, y_train)

## 計算並列出最終Loss (10分)

In [61]:
# 使用訓練資料測試模型
y_ = linear_model(x_train, theta)
loss_train = get_loss(y_, y_train)
print(f'Training Loss: {loss_train}')

# 使用測試資料測試模型
y_ = linear_model(x_test, theta)
loss_test =get_loss(y_, y_test)
print(f'Test Loss: {loss_test}')

Training Loss: 0.4987923903033696
Test Loss: 0.5937053418840041


## 二次多項式迴歸 (20分)
根據講義L2的內容，將原始的特徵$x_1$, $x_2$, $x_3$, $x_4$ 再加入二次項($x_1^2$, $x_2^2$, $x_1x_2$...)進行迴歸。<br>

完成後print出二次項迴歸模型在訓練資料跟測試資料的Loss

**Hint: 原始的x尺寸包含常數項為[N,5]，4個未知數的二次項會有10個(4+3+2+1)，加入二次項後的x應該會變為[N,15]**

In [62]:
def feature_transform(x):
    xx = x**2 # 🚧 拓展x使其成為二次多項式xx
    return xx

xx_train = feature_transform(x_train)
xx_test = feature_transform(x_test)

theta = fit(xx_train, y_train)

# 使用訓練資料測試模型
y_ = linear_model(xx_train, theta)
loss_train = get_loss(y_, y_train)
print(f'Training Loss: {loss_train}')

# 使用測試資料測試模型
y_ = linear_model(xx_train, theta)
loss_test =get_loss(y_, y_test)
print(f'Test Loss: {loss_test}')

Training Loss: 0.8156525598064536


ValueError: operands could not be broadcast together with shapes (80,1) (380,1) 